In [2]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import pickle
import os

# --- 1. Load the Dataset ---
print("Loading dataset...")
# Load the data and only keep the 'text' and 'class' columns
try:
    df_full = pd.read_csv('../data/reddit_posts.csv', encoding='latin-1')
    df = df_full[['text', 'class']].copy() # Select only the columns we need
    print(f"Dataset loaded with {len(df)} rows.")
except Exception as e:
    print(f"Error loading or processing the CSV file: {e}")


# --- 2. Preprocess and Clean the Data ---
print("\nCleaning text data...")

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Use the correct column name 'text'
df['cleaned_text'] = df['text'].apply(clean_text)

# Use the correct column name 'class' and map the labels
df['label'] = df['class'].apply(lambda x: 1 if x == 'suicide' else 0)

print("Data cleaning and labeling complete.")
print("\nSample of cleaned data:")
# Display the new columns to confirm
print(df[['text', 'cleaned_text', 'label']].head())


# --- 3. Define Features, Target, and Split Data ---
X = df['cleaned_text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"\nData split into {len(X_train)} training samples and {len(X_test)} testing samples.")


# --- 4. Create and Train the Model Pipeline ---
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('classifier', LogisticRegression(max_iter=1000))
])

print("\nTraining the model... (This may take a minute)")
model_pipeline.fit(X_train, y_train)
print("Training complete.")


# --- 5. Evaluate the Model's Performance ---
print("\nNew Model Performance:")
predictions = model_pipeline.predict(X_test)
print(classification_report(y_test, predictions))


# --- 6. Save the Trained Model ---
if not os.path.exists('../models'):
    os.makedirs('../models')

with open('../models/model_pipeline.pkl', 'wb') as f:
    pickle.dump(model_pipeline, f)

print("\n✅ Model trained and saved to models/model_pipeline.pkl")

Loading dataset...


C:\Users\91952\AppData\Local\Temp\ipykernel_25508\3780370372.py:15: DtypeWarning: Columns (0,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86) have mixed types. Specify dtype option on import or set low_memory=False.
  df_full = pd.read_csv('../data/reddit_posts.csv', encoding='latin-1')


Dataset loaded with 233372 rows.

Cleaning text data...
Data cleaning and labeling complete.

Sample of cleaned data:
                                                text  \
0  Ex Wife Threatening SuicideRecently I left my ...   
1  Am I weird I don't get affected by compliments...   
2  Finally 2020 is almost over... So I can never ...   
3          i need helpjust help me im crying so hard   
4  IÃ¢â¬â¢m so lostHello, my name is Adam (16) ...   

                                        cleaned_text  label  
0  ex wife threatening suiciderecently i left my ...      1  
1  am i weird i dont get affected by compliments ...      0  
2  finally is almost over so i can never hear has...      0  
3          i need helpjust help me im crying so hard      1  
4  im so losthello my name is adam and ive been s...      1  

Data split into 186697 training samples and 46675 testing samples.

Training the model... (This may take a minute)
Training complete.

New Model Performance:
              